In [1]:
import numpy as np
import pandas as pd
import pickle
import torch
import torch.nn as nn

REVERSE_CLASS_MAP = {
    0: 'Human', 
    1: 'OpenAI', 
    2: 'Google', 
    3: 'Meta', 
    4: 'Anthropic'
}

# Redefinir a classe da rede para o PyTorch saber carregar os pesos
class TextoClassificadorDNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(TextoClassificadorDNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

df_teste = pd.read_csv('subm1.csv', sep=';')
with open('tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

X_teste_tabular = vectorizer.transform(df_teste['Text'].values).toarray().astype(np.float32)

# Configurar e Carregar o Modelo PyTorch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_size = X_teste_tabular.shape[1]
num_classes = 5

modelo_torch = TextoClassificadorDNN(input_size, num_classes).to(device)
modelo_torch.load_state_dict(torch.load('melhor_modelo_pytorch.pth', map_location=device, weights_only=True))
modelo_torch.eval() 

# Fazer as Previsões
X_tensor = torch.tensor(X_teste_tabular).to(device)
with torch.no_grad():
    outputs = modelo_torch(X_tensor)
    _, previsoes_idx = torch.max(outputs.data, 1)

df_teste['Labels'] = [REVERSE_CLASS_MAP[idx.item()] for idx in previsoes_idx]
nome_ficheiro_saida = 'subm1-g6-MIA-B.csv'
df_teste.to_csv(nome_ficheiro_saida, index=False)

print(f"Sucesso! O ficheiro {nome_ficheiro_saida} foi gerado.")
display(df_teste.head())

Sucesso! O ficheiro subm1-g6-MIA-B.csv foi gerado.


,ID,Text,Labels
0,D2-1,A covalent bond is a chemical bond that involv...,OpenAI
1,D2-2,A covalent bond forms when two atoms share one...,OpenAI
2,D2-3,A covalent bond is a type of chemical bond whe...,OpenAI
3,D2-4,A covalent bond is a chemical bond that involv...,Human
4,D2-5,Driven by exciting developments in the field o...,Google
